## This workbook sets up an NVT simulation in graphene slit with water and NaCl. 

### The purpose of this setup is to show that :
### -- multiple force field files can be utilized (provided their 1,4-Coulombic interactions are the same)
### -- how atoms are fixed via the residue names
### -- how bonds and angles are fixed via the residue names
### -- fake waters are easily implemented to take advantage of the special Molecular Exchange Monte Carlo (MEMC) moves, which increase the simulation efficiency.  Note: they need different residue names than regular water.    

### These file writers utilize the new GOMC file writer via MoSDeF, which incorporates a GOMC force field (CHARMM style force field), psf, and pdb writers.  The main difference in these psf and pdb writers is the ability to actively select properties based on the molecule's residue, such as the force field, and fixing bond lengths and angles.  

### Build the individual molecules and minimize their energies by applying the appropriate force fields. 

In [37]:
from files.porebuilder import GraphenePore
import mbuild as mb
from foyer import Forcefield
import mbuild.formats.charmm_writer as mf_charmm

Water_mol2_file = 'files/tip3p.mol2'
Fake_water_mol2_file = 'files/fake_tip3p.mol2'

Water_res_name = 'H2O'
Fake_water_res_name = 'h2o'

FF_file_water_graphene_NaCl = 'files/FF_graphene_SPCE_NaCl.xml'
FF_file_fake_water = 'files/FF_Fake_SPCE.xml'

water = mb.load(Water_mol2_file)
water.name = Water_res_name
water.energy_minimization(forcefield = FF_file_water_graphene_NaCl  , steps=10**9)

fake_water = mb.load(Fake_water_mol2_file )
fake_water.name = Fake_water_res_name
fake_water.energy_minimization(forcefield = FF_file_fake_water , steps=10**9)


Molecule_Na = mb.Compound(name="Na")
Molecule_Na.name = 'Na'  # naming the molecule (i.e., residue)

Molecule_Cl = mb.Compound(name="Cl")
Molecule_Cl.name = 'Cl'  # naming the molecule (i.e., residue)


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/utils/decorators.py:11: DeprecationWarning: energy_minimization is deprecated. Please use Compound.energy_minimize()
  warn(printed_message, DeprecationWarning)
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:449: UserWarning: No force field version number found in force field XML file.
  'No force field version number found in force field XML file.'
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:461: UserWarning: No force field name found in force 

### Enter the data required to build the water-salt in a graphene slit in the GOMC format

In [39]:
FF_Graphene_pore_w_solvent_Dict = {Molecule_Na.name : FF_file_water_graphene_NaCl,
                                   Molecule_Cl.name : FF_file_water_graphene_NaCl,
                                   water.name: FF_file_water_graphene_NaCl ,
                                   fake_water.name: FF_file_fake_water,
                                   'BOT' : FF_file_water_graphene_NaCl ,
                                   'TOP' : FF_file_water_graphene_NaCl
                                  }

residues_Graphene_pore_w_solvent_List = [ Molecule_Na.name,
                                         Molecule_Cl.name,
                                          water.name,
                                         fake_water.name,
                                         'BOT', 'TOP']

Fix_bonds_angles_residues = [ water.name, fake_water.name]

Fix_Graphene_residue = [ 'BOT', 'TOP']

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Create the graphene slit with water, fake water, and NaCl.  

### Note: the fake water is limited to 5 molecules, so it does not dramatically affect the results, as to it is has different force field parameters.  

### Note: The graphene pore was modified from the existing setup (i.e., graphene sheet on the beginning and end of the z-axis).  This change allows the atoms to be identically represented after the GOMC simulation without manually adjusting them in VMD.  This change will fundamentally be the same as the original configuration, as the other side of the graphene sheet is now just through the periodic boundary of the z-axis.

In [40]:
pore_width_nm = 2.0
No_sheets = 3
sheet_spacing = 0.335

water_spacing_from_walls = 0.2

Total_molecules = 500
n_fake_waters = 5
n_Na = 10
n_Cl = 10

#for GOMC, currently we need to add the space at the end of the simulation
# this does not matter as we are using PBC's
empty_graphene_pore = GraphenePore(
        pore_width = sheet_spacing ,
        pore_length = 3.0,
        pore_depth = 3.0,
        n_sheets = No_sheets,
        slit_pore_dim = 2
)

empty_graphene_pore_shifted = empty_graphene_pore


n_waters = Total_molecules - n_fake_waters - n_Na - n_Cl

#note the default spacing of 0.2 automatically accounted for in the water box packing (i.e. adding 0.2 nm for 1 wall is really 0.4 nm)
water_between_pores = mb.fill_box(compound=[water,fake_water, Molecule_Na, Molecule_Cl ],
                                  n_compounds= [n_waters, n_fake_waters, n_Na, n_Cl] ,
                                  box=[3.0, 3.0, pore_width_nm - water_spacing_from_walls*1])
water_between_pores.translate([0,  0, sheet_spacing*(2*No_sheets-1) + water_spacing_from_walls])

filled_pore = empty_graphene_pore
filled_pore.add(water_between_pores, inherit_periodicity=False)
filled_pore.translate([ -filled_pore.center[0],   -filled_pore.center[1], 0])
filled_pore.periodicity[2] = sheet_spacing*(2*No_sheets-1)+pore_width_nm

filled_pore.visualize()

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


replicate = [12, 14.104648270104862]


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/lattice.py:626: UserWarning: Periodicity of non-rectangular lattices are not valid with default boxes. Only rectangular lattices are valid at this time.
  warn('Periodicity of non-rectangular lattices are not valid with '


You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

### Write the GOMC force field, psf, and pdb files.  

### This writer only uses 1 box since it is an NVT simulation.  Therefore, the   structure_1 and filename_1 are not included (i.e., equals None).  However, many simulations do not have all the atom types or molecules in all the simulation boxes (Example: Gibbs and grand-canonical ensembles). 

### Note: the forcefield_files variable sets the individual residues to their own force fields.  In other words, molecules can use a specific forcefield by custom naming their residues.

### Note: the fix_residue variable fixed the carbon molecules in the graphene sheet so they do not move.  

### Note: the fix_res_bonds_angles fixes the waters and fake waters bond lengths and angles.  

### Note: the extensions of the psf/pdb files ('filled_pore_water_fake_water_NaCl_3x3x2.0nm_3-layer') are note entered as they are automatically added.

In [42]:
print('Running: GOMC FF file, and the psf and pdb files')
mf_charmm.charmm_psf_psb_FF(filled_pore,
                            'filled_pore_water_fake_water_NaCl_3x3x2.0nm_3-layer',
                            structure_1 =None ,
                            filename_1 = None,
                            GOMC_FF_filename ="GOMC_pore_water_fake_water_NaCl_FF" ,
                            forcefield_files = FF_Graphene_pore_w_solvent_Dict,
                            residues= residues_Graphene_pore_w_solvent_List ,
                            Bead_to_atom_name_dict = None,
                            fix_residue = Fix_Graphene_residue,
                            fix_res_bonds_angles = Fix_bonds_angles_residues,
                            reorder_res_in_pdb_psf = False
                            )
print('Completed: GOMC FF file, and the psf and pdb files')

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Running: GOMC FF file, and the psf and pdb files
write_gomcdata: forcefield_files = {'Na': 'files/FF_graphene_SPCE_NaCl.xml', 'Cl': 'files/FF_graphene_SPCE_NaCl.xml', 'H2O': 'files/FF_graphene_SPCE_NaCl.xml', 'h2o': 'files/FF_Fake_SPCE.xml', 'BOT': 'files/FF_graphene_SPCE_NaCl.xml', 'TOP': 'files/FF_graphene_SPCE_NaCl.xml'}, residues = ['Na', 'Cl', 'H2O', 'h2o', 'BOT', 'TOP']
INFORMATION: The following residues will have fixed bonds and angles: fix_res_bonds_angles = ['H2O', 'h2o']
******************************

GOMC FF writing each residues FF as a group for structure_0


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/parmed/openmm/topsystem.py:238: OpenMMWarning: Adding what seems to be Urey-Bradley terms before Angles. This is unexpected, but the parameters will all be present in one form or another.
  'all be present in one form or another.', OpenMMWarning)
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/parmed/openmm/topsystem.py:238: OpenMMWarning: Adding what seems to be Urey-Bradley terms before Angles. This is unexpected, but the parameters will all be present in one form or another.
  'all be present in one form or another.', OpenMMWarning)
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/parmed/openmm/topsystem.py:238: OpenMMWarning: Adding what seems to be Urey-Bradley terms before Angles. This is unexpected, but the parameters will all be present in one form or another.
  'all be present in one form or another.', OpenMMWarning)
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/pyt

forcefield type from compound = {'Na': 'files/FF_graphene_SPCE_NaCl.xml', 'Cl': 'files/FF_graphene_SPCE_NaCl.xml', 'H2O': 'files/FF_graphene_SPCE_NaCl.xml', 'h2o': 'files/FF_Fake_SPCE.xml', 'BOT': 'files/FF_graphene_SPCE_NaCl.xml', 'TOP': 'files/FF_graphene_SPCE_NaCl.xml'}
coulomb14scale from compound = {'Na': 0.5, 'Cl': 0.5, 'H2O': 0.5, 'h2o': 0.5, 'BOT': 0.5, 'TOP': 0.5}
lj14scale from compound = {'Na': 0.5, 'Cl': 0.5, 'H2O': 0.5, 'h2o': 0.5, 'BOT': 0.5, 'TOP': 0.5}
No urey bradley terms detected, will use angle_style harmonic
******************************

writing the GOMC force field file 
NBFIX_Mixing not used or no mixing used for the non-bonded potentials out

******************************

write_charmm_psf file is running
write_charmm_psf: forcefield_files = {'Na': 'files/FF_graphene_SPCE_NaCl.xml', 'Cl': 'files/FF_graphene_SPCE_NaCl.xml', 'H2O': 'files/FF_graphene_SPCE_NaCl.xml', 'h2o': 'files/FF_Fake_SPCE.xml', 'BOT': 'files/FF_graphene_SPCE_NaCl.xml', 'TOP': 'files/FF_grap